**Q1. L4 Quiz Review**

**a) Step 1: Re-examining Codes and Results**

The code runs as expected and I cross-checked the results with the AI.

**b) Step 2: Optimization & Best Practices**

Efficiency and readability:
* The looping for the rolling regression was inefficient as it ran the OLS model 252 x N times, which rebuilds the same slices repeatedly and is computationally expensive for larger N.
* Ideally, there should be data cleaning processes, but the step was skipped because it was confirmed that there are no NaN values when calling .info().
* Redundant slicing inside the loop (window_data = merged.iloc[i - window:i]), which is recomputed every iteration for each X slice, Y slice, and constant addition.
* Not the most readable as all processes are embedded in one loop, which makes it hard to debug, reuse, or extend.

Areas for improvement:
* Improve efficiency using Numpy-based windowing.
* Clean the structure to promote readability and reusability of the code (add comments).
* Clean data regardless of no NaNs (convert everything early on to avoid confusion and mismatched data types).


In [8]:
#REFACTORED CODE

import pandas as pd
import statsmodels.api as sm
crsp = pd.read_csv("crsp_roll.csv")
#crsp.info()
#print(crsp.columns)
crsp.head()

ff = pd.read_csv("ff_daily.csv")
#ff.info()
#print(ff.columns)
ff.head()

#1. Data Cleaning --------------------------------------------------------------

crsp['date'] = pd.to_datetime(crsp['date'])

ff['date'] = pd.to_datetime(ff['index'], format='%Y%m%d')
ff = ff.drop(columns=['index'])

# sort early (important for rolling)
crsp = crsp.sort_values(['permno', 'date']).reset_index(drop=True)

# filter stock
permno = 23473
stock = crsp.loc[crsp['permno'] == permno].copy()

# merge
merged = stock.merge(ff, on='date', how='inner')

merged = merged.rename(columns={
    'Mkt-RF': 'mktexrf',
    'SMB': 'smb',
    'HML': 'hml',
    'RF': 'rf'
})

# excess return
merged['retxrf'] = merged['ret'] - merged['rf']
# drop missing early
merged = merged.dropna(subset=['retxrf', 'mktexrf', 'smb', 'hml'])
# scale before rolling
scale_cols = ['retxrf', 'mktexrf', 'smb', 'hml']
merged[scale_cols] = merged[scale_cols] / 100

#print(merged.head())

#2. Pre-compute arrays ---------------------------------------------------------

Y = merged['retxrf'].values
X = merged[['mktexrf', 'smb', 'hml']].values
dates = merged['date'].values

window = 252

alphas, betas_mkt, betas_smb, betas_hml, out_dates = [], [], [], [], []

#3. Rolling regression ---------------------------------------------------------

for i in range(window, len(merged)):

    y_window = Y[i-window:i]
    x_window = X[i-window:i]

    X_design = sm.add_constant(x_window)
    model = sm.OLS(y_window, X_design).fit()

    alphas.append(model.params[0])
    betas_mkt.append(model.params[1])
    betas_smb.append(model.params[2])
    betas_hml.append(model.params[3])
    out_dates.append(dates[i-1])

#4. Output DataFrame -----------------------------------------------------------

rolling_df = pd.DataFrame({
    'date': out_dates,
    'alpha': alphas,
    'beta_mkt': betas_mkt,
    'beta_smb': betas_smb,
    'beta_hml': betas_hml
})

rolling_df.head()

,date,alpha,beta_mkt,beta_smb,beta_hml
0,2024-01-03,-0.000201,0.009997,-0.000895,0.006285
1,2024-01-04,-0.000201,0.010028,-0.000901,0.006311
2,2024-01-05,-0.000202,0.009987,-0.000960,0.006314
3,2024-01-08,-0.000202,0.010006,-0.000992,0.006301
4,2024-01-09,-0.000203,0.009913,-0.000939,0.006293


**c) Step 3: Review Transcript of Conversation with AI**
* Efficiency Analysis:

It took me a total of 12 prompts to get a working solution. I did have to correct the AI on the same mistakes, such as what columns there are, that its supposed to merge weekly/daily, that there are inconsistencies in time measurement which may cause the merge to come out empty, etc. To be fair, it would have been more efficient for me to copy-paste the quiz to provide all the context that it needs, but honestly that felt like cheating, so I took my time to actually explore how to compute the rolling regression for the three-factor model. Most of my prompts were sending the AI feedback on what errors were popping up.

* Critical Evaluation of AI Output:

Throughout the quiz I did accept the initial code, then I checked by .head()-ing the merged dataset, and it was empty because it wasn't merging properly based on day/week. The AI also did not realize that it didn't rescale the returns, so the initial results had really large alphas. It was not until I sent back the code output to the AI and asked if it was reasonable, did it realize the metrics weren't quite correct.

* Future Strategy:

I would copy-paste the question, then ask it to code it step-by-step and provide explanations per step so I can understand the process and debug per step. I think when they produce the full code, people tend to get tempted to just blindly trust the output without understanding anything or realizing the AI can make errors. Before asking the AI to generate code, I would include the output of the files' .head(), so it will reduce the chances of column name mismatch, and it will know if the data was weekly/daily. An example perfect prompt would be: "[copy-paste of quiz] [copy-paste of .head()] can you please show me how to code this step-by-step".